# 05. 무대 위에서 — **통합**

## 이 노트북의 위치

```
기반     : NB01 (채보·분석)
협업     : NB02 (Somax) + NB03 (RAVE)
확장     : NB04 (시각)
통합 ✦   : NB05 (무대) ← 여기
```

확장과 협업의 모든 산출물을 **하나의 미니 공연으로 엮습니다**.

## 재료
- 원 연주 2곡 (Satie, Prokofiev)
- NB04 학생의 cue 매핑 (확장: 시각)
- NB03 stage extension 메타데이터 (선택)

## 학생이 하는 것
**타임라인을 직접 편집해서** 공연의 흐름을 설계합니다.
어느 곡을 언제, 어떤 매핑으로 보여줄지 결정합니다.

`/stage` 페이지에서는 **재생 모드**(녹음 재생)와 **라이브 추적 모드**(matchmaker 연동)를 전환할 수 있습니다.

Colab 단독 실행 시에는 웹 폴더 동기화 대신 `stage_data_export.zip`으로 내보낼 수 있습니다.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

import librosa
import numpy as np

IS_COLAB = 'google.colab' in sys.modules
RUN_MODE = os.environ.get('PIANOKIT_RUN_MODE', 'colab' if IS_COLAB else 'server')
REPO_URL = os.environ.get('PIANOKIT_REPO_URL', 'https://github.com/jhbae/pianokit.git')

if IS_COLAB and RUN_MODE == 'colab' and not Path('assets').exists():
    repo_dir = Path('/content/pianokit')
    if not repo_dir.exists():
        subprocess.run(['git', 'clone', REPO_URL, str(repo_dir)], check=True)
    os.chdir(str(repo_dir))
    print(f'[bootstrap] changed cwd to {repo_dir}')

if IS_COLAB and RUN_MODE == 'colab':
    subprocess.run([
        sys.executable, '-m', 'pip', 'install', '-q',
        'librosa', 'numpy', 'matplotlib', 'soundfile', 'ipython'
    ], check=False)

ARTIFACTS = Path(os.environ.get('PIANOKIT_ARTIFACTS_DIR', 'artifacts'))
ASSETS = Path(os.environ.get('PIANOKIT_ASSETS_DIR', 'assets'))

# required outputs
required = {
    'satie_audio': ASSETS / 'satie_gymnopedie_no1.wav',
    'prokofiev_audio': ASSETS / 'prokofiev_toccata_op11.wav',
    'satie_cues': ARTIFACTS / 'cues' / 'satie_visual_cues.json',
    'prokofiev_cues': ARTIFACTS / 'cues' / 'prokofiev_visual_cues.json',
}
optional = {
    'alignment_satie': ARTIFACTS / 'alignment' / 'satie_alignment.json',
    'alignment_prokofiev': ARTIFACTS / 'alignment' / 'prokofiev_alignment.json',
    'collab_stage_extension': ARTIFACTS / 'collab_rave' / 'stage_extension.json',
    'collab_style_metadata': ARTIFACTS / 'collab_rave' / 'style_metadata.json',
}

for cue_path in [required['satie_cues'], required['prokofiev_cues']]:
    cue_path.parent.mkdir(parents=True, exist_ok=True)

def build_basic_cues(audio_path: Path, cue_path: Path, piece: str, fps: int = 20):
    y, sr = librosa.load(str(audio_path), sr=22050, mono=True)
    hop = max(1, int(sr / fps))

    rms = librosa.feature.rms(y=y, hop_length=hop)[0]
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop)[0]
    rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr, hop_length=hop)[0]
    onset_env = librosa.onset.onset_strength(y=y, sr=sr, hop_length=hop)

    n_frames = min(len(rms), len(centroid), len(rolloff), len(onset_env))
    if n_frames == 0:
        raise RuntimeError(f'Failed to build cues from {audio_path}')

    def norm(x):
        x = np.asarray(x[:n_frames], dtype=float)
        lo, hi = float(np.min(x)), float(np.max(x))
        if hi - lo < 1e-8:
            return np.zeros_like(x)
        return (x - lo) / (hi - lo)

    energy = norm(rms)
    brightness = norm(centroid)
    tension = norm(rolloff)
    onset = norm(onset_env)

    frames = []
    for i in range(n_frames):
        frames.append({
            't': i / fps,
            'energy': float(energy[i]),
            'density': float((energy[i] + onset[i]) * 0.5),
            'onset_density': float(onset[i]),
            'register': float(brightness[i]),
            'brightness': float(brightness[i]),
            'attack': float(onset[i]),
            'register_spread': float((brightness[i] + tension[i]) * 0.5),
            'harmonic_tension': float(tension[i]),
            'velocity_variance': float(abs(onset[i] - energy[i])),
            'pulse': float((onset[i] + energy[i]) * 0.5),
        })

    payload = {
        'piece': piece,
        'audio_file': str(audio_path),
        'duration': float(len(y) / sr),
        'fps': fps,
        'frames': frames,
    }
    cue_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2))
    print(f'[fallback] generated cues: {cue_path}')

print(f'Run mode: {RUN_MODE} (colab={IS_COLAB})')
print('Required artifacts:')
for k, p in required.items():
    ok = 'OK' if p.exists() else 'MISSING'
    print(f'  {ok:7s} {k}: {p}')

print('\nOptional artifacts (NB03):')
for k, p in optional.items():
    ok = 'OK' if p.exists() else 'MISSING'
    print(f'  {ok:7s} {k}: {p}')

missing_audio = [k for k in ['satie_audio', 'prokofiev_audio'] if not required[k].exists()]
if missing_audio:
    raise FileNotFoundError(
        f'Missing audio files: {missing_audio}. '
        'Set PIANOKIT_ASSETS_DIR or clone repo data first.'
    )

# Standalone fallback: auto-generate minimal cues when absent.
if not required['satie_cues'].exists():
    build_basic_cues(required['satie_audio'], required['satie_cues'], 'satie')
if not required['prokofiev_cues'].exists():
    build_basic_cues(required['prokofiev_audio'], required['prokofiev_cues'], 'prokofiev')

---
## 1. 공연 타임라인 편집

아래 `timeline` 리스트를 직접 편집하세요. 각 섹션은 **source + mapping + optional text overlay**입니다.

### source 종류
- `satie`, `prokofiev`: 원 연주

### mapping 종류
- `서정적`, `역동적`, `미니멀`: NB04 프리셋
- `custom`: 직접 정의

In [ ]:
# Edit this timeline as needed.
timeline = [
    {
        'section': 'Intro',
        'source': 'satie',
        'start': 0.0,
        'duration': 45.0,
        'mapping': 'lyrical',
        'text_overlay': None,
    },
    {
        'section': 'Transition',
        'source': 'satie',
        'start': 45.0,
        'duration': 15.0,
        'mapping': 'minimal',
        'text_overlay': 'Mapping change: lyrical -> minimal',
    },
    {
        'section': 'Burst',
        'source': 'prokofiev',
        'start': 0.0,
        'duration': 60.0,
        'mapping': 'dynamic',
        'text_overlay': None,
    },
    {
        'section': 'Contrast',
        'source': 'prokofiev',
        'start': 60.0,
        'duration': 20.0,
        'mapping': 'lyrical',
        'text_overlay': 'Mapping change: dynamic -> lyrical',
    },
]

total = sum(s['duration'] for s in timeline)
print(f'Total duration: {total:.1f}s ({total/60:.1f}min)')
print()
print('Section layout:')
cursor = 0.0
for s in timeline:
    print(f"  {cursor:5.1f}s -> {cursor + s['duration']:5.1f}s | {s['section']:10s} | {s['source']:10s} | mapping={s['mapping']:12s}" + (f" | {s['text_overlay']}" if s['text_overlay'] else ''))
    cursor += s['duration']

---
## 1-b. Auto-add NB03 RAVE sections (optional)

If `artifacts/collab_rave/*.wav` exists, this cell can append sections to `timeline` automatically.
Each generated section uses a dedicated `source` key (`rave_<profile>`), so it can be exported with the performance JSON.

In [ ]:
import soundfile as sf

auto_add_rave_sections = True
rave_section_duration_cap = 20.0  # seconds per inserted section

rave_dir = ARTIFACTS / 'collab_rave'
rave_wavs = sorted(rave_dir.glob('*.wav')) if rave_dir.exists() else []
rave_sources = {}

for wav_path in rave_wavs:
    stem = wav_path.stem
    # expected format: <piece>_<profile>
    if '_' not in stem:
        continue
    piece, profile = stem.split('_', 1)
    cue_key = f'{piece}_cues'
    if cue_key not in required:
        continue

    info = sf.info(str(wav_path))
    duration = float(info.frames / info.samplerate) if info.samplerate > 0 else 0.0
    source_key = f'rave_{profile}'

    rave_sources[source_key] = {
        'audio': wav_path,
        'cues': required[cue_key],
        'piece': piece,
        'profile': profile,
        'duration': duration,
    }

if auto_add_rave_sections and rave_sources:
    existing_sections = {s['section'] for s in timeline}
    for source_key, meta in rave_sources.items():
        section_name = f"RAVE-{meta['profile']}"
        if section_name in existing_sections:
            continue
        timeline.append({
            'section': section_name,
            'source': source_key,
            'start': 0.0,
            'duration': min(rave_section_duration_cap, meta['duration']),
            'mapping': 'custom',
            'text_overlay': f"NB03 style: {meta['profile']}",
        })

# recalc summary after auto-append
total = sum(s['duration'] for s in timeline)
print(f'RAVE sources detected: {len(rave_sources)}')
for k, v in rave_sources.items():
    print(f"  {k}: {v['audio'].name} ({v['duration']:.2f}s)")

print(f'Updated total duration: {total:.1f}s ({total/60:.1f}min)')
print('Updated section layout:')
cursor = 0.0
for s in timeline:
    print(f"  {cursor:5.1f}s -> {cursor + s['duration']:5.1f}s | {s['section']:16s} | {s['source']:18s} | mapping={s['mapping']:12s}" + (f" | {s['text_overlay']}" if s['text_overlay'] else ''))
    cursor += s['duration']

---
## 2. 타임라인 시각화

각 섹션의 cue 곡선을 이어 붙여 공연 전체의 흐름을 미리 봅니다.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager, rcParams

# Configure a Korean-capable plotting font when available.
def configure_plot_font():
    candidates = [
        'Noto Sans CJK JP',
        'NanumGothic',
        'AppleGothic',
        'Malgun Gothic',
    ]
    for name in candidates:
        try:
            path = font_manager.findfont(name, fallback_to_default=False)
            rcParams['font.family'] = name
            rcParams['axes.unicode_minus'] = False
            print(f'Plot font set to: {name} ({path})')
            return
        except Exception:
            pass

    rcParams['font.family'] = 'DejaVu Sans'
    rcParams['axes.unicode_minus'] = False
    print('Plot font fallback: DejaVu Sans (Korean text may not render fully).')

configure_plot_font()

cues_cache = {
    'satie': json.loads(required['satie_cues'].read_text()),
    'prokofiev': json.loads(required['prokofiev_cues'].read_text()),
}

fps = cues_cache['satie']['fps']
channels = ['energy', 'density', 'harmonic_tension', 'register_spread']
colors = {'satie': '#4f46e5', 'prokofiev': '#dc2626'}

fig, axes = plt.subplots(len(channels), 1, figsize=(14, 8), sharex=True)
global_t = 0.0
for section in timeline:
    src = section['source']
    if src not in cues_cache:
        continue
    frames = cues_cache[src]['frames']
    start_frame = int(section['start'] * fps)
    end_frame = int((section['start'] + section['duration']) * fps)
    segment = frames[start_frame:end_frame]
    seg_t = np.arange(len(segment)) / fps + global_t
    for ax, ch in zip(axes, channels):
        vals = [f[ch] for f in segment]
        ax.fill_between(seg_t, vals, alpha=0.6, color=colors.get(src, '#888'))
    for ax in axes:
        ax.axvline(global_t, color='black', linestyle='--', alpha=0.3, linewidth=0.8)
    axes[0].text(global_t + 0.3, 0.95, section['section'], fontsize=10, fontweight='bold', transform=axes[0].get_xaxis_transform())
    global_t += section['duration']

for ax, ch in zip(axes, channels):
    ax.set_ylabel(ch)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.2)

axes[-1].set_xlabel('Performance time (s)')
axes[0].set_title('Performance timeline - cue flow')
plt.tight_layout()
plt.show()

---
## 3. 공연 JSON 내보내기

웹 앱(`pianokit_web/`)의 `/stage` 라우트에서 이 JSON을 소비합니다.

In [ ]:
performance = {
    'title': 'pianokit performance - Satie x Prokofiev',
    'total_duration': total,
    'timeline': timeline,
    'sources': {
        'satie': {
            'audio': str(required['satie_audio']),
            'cues': str(required['satie_cues']),
            'label': 'Satie',
            'group': 'original',
            'order': 10,
            'piece': 'satie',
        },
        'prokofiev': {
            'audio': str(required['prokofiev_audio']),
            'cues': str(required['prokofiev_cues']),
            'label': 'Prokofiev',
            'group': 'original',
            'order': 20,
            'piece': 'prokofiev',
        },
    },
    'claude_mapping_suggestions': {},
}

# Merge optional RAVE sources auto-detected in 1-b.
extra_sources = globals().get('rave_sources', {})
rave_order_base = 100
for idx, (source_key, meta) in enumerate(sorted(extra_sources.items())):
    profile_label = str(meta.get('profile', source_key.replace('rave_', ''))).replace('_', ' ').title()
    performance['sources'][source_key] = {
        'audio': str(meta['audio']),
        'cues': str(meta['cues']),
        'label': f'RAVE {profile_label}',
        'group': 'rave',
        'order': rave_order_base + idx,
        'piece': str(meta.get('piece', 'satie')),
    }
if extra_sources:
    print(f"Added RAVE sources: {', '.join(extra_sources.keys())}")

stage_ext_path = optional.get('collab_stage_extension')
style_meta_path = optional.get('collab_style_metadata')
if stage_ext_path and stage_ext_path.exists():
    performance['collab_stage_extension'] = json.loads(stage_ext_path.read_text())
    print(f"Merged NB03 stage_extension: {stage_ext_path}")
if style_meta_path and style_meta_path.exists():
    performance['collab_style_metadata_file'] = str(style_meta_path)
    print(f"Added NB03 style_metadata reference: {style_meta_path}")

out_path = ARTIFACTS / 'performance_timeline.json'
out_path.write_text(json.dumps(performance, ensure_ascii=False, indent=2))
print(f'Saved performance JSON: {out_path}')
print(f"  Sources: {', '.join(performance['sources'].keys())}")
print(f'  Sections: {len(timeline)}, total {total:.1f}s')

---
## 4. 웹 앱 연동

`pianokit_web/public/performance_timeline.json`으로 복사하면 웹 앱이 소비할 수 있습니다.

In [ ]:
import shutil
import zipfile

web_public = Path('pianokit_web') / 'public'
if web_public.exists():
    web_data = web_public / 'stage_data'
    web_data.mkdir(exist_ok=True)
    shutil.copy(out_path, web_data / 'performance_timeline.json')
    for key in ['satie', 'prokofiev']:
        shutil.copy(required[f'{key}_audio'], web_data / f'{key}.wav')
        shutil.copy(required[f'{key}_cues'], web_data / f'{key}_cues.json')

    for key in ['satie', 'prokofiev']:
        align_file = optional.get(f'alignment_{key}')
        if align_file and align_file.exists():
            shutil.copy(align_file, web_data / f'{key}_alignment.json')
            print(f'  Copied alignment for {key}')

    if optional.get('collab_stage_extension') and optional['collab_stage_extension'].exists():
        shutil.copy(optional['collab_stage_extension'], web_data / 'collab_stage_extension.json')
        print('  Copied NB03 stage_extension')
    if optional.get('collab_style_metadata') and optional['collab_style_metadata'].exists():
        shutil.copy(optional['collab_style_metadata'], web_data / 'collab_style_metadata.json')
        print('  Copied NB03 style_metadata')

    collab_rave_dir = ARTIFACTS / 'collab_rave'
    for wav_path in collab_rave_dir.glob('*.wav'):
        shutil.copy(wav_path, web_data / wav_path.name)
        print(f'  Copied NB03 style audio: {wav_path.name}')

    print(f'Copied stage data to: {web_data}')
    print('Access in web app with fetch("/stage_data/performance_timeline.json")')
else:
    # Colab-friendly export path
    export_dir = ARTIFACTS / 'colab_stage_export'
    export_dir.mkdir(parents=True, exist_ok=True)

    files_to_copy = [out_path, required['satie_audio'], required['prokofiev_audio'], required['satie_cues'], required['prokofiev_cues']]
    for p in files_to_copy:
        if p.exists():
            shutil.copy(p, export_dir / p.name)

    for key in ['collab_stage_extension', 'collab_style_metadata', 'alignment_satie', 'alignment_prokofiev']:
        p = optional.get(key)
        if p and p.exists():
            shutil.copy(p, export_dir / p.name)

    collab_rave_dir = ARTIFACTS / 'collab_rave'
    if collab_rave_dir.exists():
        for wav_path in collab_rave_dir.glob('*.wav'):
            shutil.copy(wav_path, export_dir / wav_path.name)

    zip_path = ARTIFACTS / 'stage_data_export.zip'
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
        for p in sorted(export_dir.glob('*')):
            zf.write(p, arcname=p.name)

    print(f'{web_public} not found. Created Colab export zip: {zip_path}')
    print('Download this zip and copy files into pianokit_web/public/stage_data on server.')

---
## 공연 체크리스트

조별 발표 전 확인:

1. ✅ 타임라인의 각 섹션이 음악적 흐름을 가지는가?
2. ✅ 각 구간의 매핑 선택이 음악 성격과 맞는가? (확장의 품질)
3. ✅ 두 곡의 대비가 공연 전체의 드라마를 만드는가?
4. ✅ 공연 흐름이 자연스러운가?
5. ✅ NB03 스타일 확장 메타(`collab_stage_extension`)가 timeline JSON에 병합되었는가?

## 확장 과제

- 같은 구간에서 **다른 매핑 프리셋**을 비교하는 A/B 섹션 만들기
- `/stage` 페이지에서 라이브 추적 모드로 직접 연주하며 시연
- 조끼리 타임라인을 교환해서 다시 편집

---

## 워크숍 마무리 질문

- **확장**: 내 연주의 어떤 차원이 가장 설득력 있게 시각화되었는가?
- **협업**: matchmaker가 내 루바토를 제대로 따라갔는가? 어느 곡이 더 어려웠는가?